[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NguyenVu04/templates/blob/main/machine-learning/notebooks/01_clean_and_split.ipynb)

# 01 — Cleaning and Splitting

**Purpose.** Turn raw data into clean train and test splits, using only
deterministic, model-agnostic rules.

**Inputs.** `data/raw/` and the findings from notebook 00.

**Outputs.** `data/processed/train.parquet`, `data/processed/test.parquet`,
both DVC-tracked.

---

### The boundary rule — the most important rule in this project

**Allowed here** (deterministic, model-agnostic):
- Schema and dtype validation
- Duplicate removal
- Dropping non-predictive columns (record IDs and the like)
- Dropping records with a missing or invalid label
- Dropping records that violate a **hard, externally known constraint**
- The train/test split

**Forbidden here** (learns from the data — belongs in `02x`):
- Imputation of missing values
- Statistical outlier detection (IQR, z-score, isolation forest)
- Scaling, encoding, any fitted transformation
- Feature engineering and selection

| Type | Criterion | Where |
|---|---|---|
| Invalid record | Violates a bound known *before* seeing the data | **01** |
| Statistical outlier | Threshold derived *from* the data | **02x** |

Any data-derived threshold applied here is computed over train *and* test, and
leaks test-set information into the filtering decision.

**This notebook is a thin wrapper over `src/data/clean.py`.** The same sequence
runs unattended as `task clean:data` and as the `clean_split` DVC stage. Put
logic in the module, narration here.

## 0. Environment

Run this section first, wherever you are.

**Locally** it only walks up to the project root and makes it the working
directory, so the root-relative paths in `configs/data.yaml` resolve the same
way they do for `task clean:data` and the DVC pipeline. Nothing is installed.

**In Colab** it also clones the repository, puts it on `sys.path` so `import src`
works without an editable install, and installs the few packages Colab does not
already ship. Note that `data/` and `models/` are DVC-tracked and therefore *not*
part of the clone — a fresh runtime has neither. See the Drive cell below.

In [ ]:
# --- Environment bootstrap -------------------------------------------------
# Identical in every notebook. Forked the template? Change these three values
# and the badge URL at the top of this notebook.
REPO_URL = "https://github.com/NguyenVu04/templates.git"
BRANCH = "main"
SUBDIR = "machine-learning"  # project root inside the repository

# (import name, pip name). Colab already ships numpy, pandas, pyarrow,
# scikit-learn, joblib, matplotlib and seaborn, so only these are installed —
# which keeps the bootstrap fast and avoids a "restart runtime" prompt.
COLAB_PACKAGES = [("hydra", "hydra-core")]

import importlib.util
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    checkout = Path("/content") / Path(REPO_URL).stem
    if not checkout.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(checkout)],
            check=True,
        )
    root = checkout / SUBDIR
    missing = [pip for mod, pip in COLAB_PACKAGES if importlib.util.find_spec(mod) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
else:
    # JupyterLab starts the kernel in notebooks/; walk up to the project root.
    root = Path.cwd()
    while not (root / "pyproject.toml").exists() and root != root.parent:
        root = root.parent

os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))  # makes `import src` work without an editable install

# Extra Hydra overrides consumed by load_config() in section 1. Empty unless
# the Drive cell below fills it in, so local runs are unaffected.
CONFIG_OVERRIDES: list[str] = []

print(f"project root: {root}   colab: {IN_COLAB}")

In [ ]:
# --- Colab: data and artifacts (optional) ----------------------------------
# data/ and models/ are DVC-tracked, so they are not in the Git clone and a
# fresh Colab runtime has neither. Mount Drive and point the config at it —
# Drive also survives a runtime reset, which /content does not. This notebook
# *writes* the splits, so without Drive they are lost when the runtime ends.
#
# from google.colab import drive
#
# drive.mount("/content/drive")
# DATA_ROOT = "/content/drive/MyDrive/<project-name>/data"
# CONFIG_OVERRIDES += [
#     f"raw_path={DATA_ROOT}/raw/dataset.parquet",
#     f"train_path={DATA_ROOT}/processed/train.parquet",
#     f"test_path={DATA_ROOT}/processed/test.parquet",
# ]

## 1. Setup

In [ ]:
# Standard setup for every notebook in this project.
# Autoreload so edits in src/ take effect without restarting the kernel.
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.config import load_config
from src.utils.plotting import setup_plotting
from src.utils.seed import set_seed

cfg = load_config(overrides=CONFIG_OVERRIDES)  # empty unless section 0 filled it in
set_seed(cfg.seed)
setup_plotting()  # matplotlib/seaborn styling for report-ready figures

pd.set_option("display.max_columns", 50)
cfg

## 2. Load the raw data

In [ ]:
from src.data.load import load_raw

df_raw = load_raw(cfg)
audit = [("raw", len(df_raw))]
print(f"{len(df_raw):,} records loaded")
df_raw.head()

## 3. Validate against the declared schema

Fail here rather than three notebooks later. `configs/data.yaml` declares the
contract; `src/data/schema.py` enforces it.

In [ ]:
from src.data import schema

df = schema.validate(df_raw, cfg, strict=False)
schema.find_violations(df, cfg).head(20)

## 4. Remove duplicates

Decide explicitly what "duplicate" means for this dataset — fully identical
rows, or rows identical on a natural key — and record the choice.

In [ ]:
from src.data.clean import drop_duplicates

df = drop_duplicates(df, cfg)
audit.append(("after dedup", len(df)))
print(f"{audit[-2][1] - len(df):,} duplicate records removed")

## 5. Drop records with a missing or invalid label

Unlabelled records cannot be trained or scored on, so dropping them is
model-agnostic. Report the share: a large fraction is a data-collection problem
worth surfacing, not something to quietly discard.

In [ ]:
from src.data.clean import drop_invalid_labels

df = drop_invalid_labels(df, cfg)
audit.append(("after label filter", len(df)))
print(f"{audit[-2][1] - len(df):,} records dropped for missing/invalid labels")

## 6. Apply hard constraints

⚠️ **Only bounds with a documented external source.** Each rule applied here is
declared in `configs/data.yaml` with its `source` field: the standard,
specification or physical limit that defines it.

If you are about to write `df[df.x < df.x.quantile(0.99)]`, stop — that is a
statistical threshold. It belongs in a `02x` notebook, fitted on train only.

In [ ]:
from src.data.clean import drop_constraint_violations

df = drop_constraint_violations(df, cfg)
audit.append(("after hard constraints", len(df)))
print(f"{audit[-2][1] - len(df):,} records violated a hard constraint")

## 7. Drop non-predictive columns

Identifiers let a model memorise records and often encode collection order.
Drop them after deduplication, which may need them — and keep the grouping
column until after the split.

In [ ]:
from src.data.clean import drop_non_predictive

df = drop_non_predictive(df, cfg)
df.columns.tolist()

## 8. Cleaning audit

Row counts per step. This table is the evidence that cleaning did what section
10 of notebook 00 specified — paste it into the report.

In [ ]:
audit_df = pd.DataFrame(audit, columns=["step", "records"])
audit_df["removed"] = -audit_df["records"].diff().fillna(0).astype(int)
audit_df["share_remaining"] = audit_df["records"] / audit[0][1]
audit_df

## 9. Re-validate

The same contract, checked against the cleaned frame. Passing here proves the
cleaning code actually enforced what the config declares.

In [ ]:
df = schema.validate(df, cfg, strict=True)
print("schema OK")

## 10. Train/test split

From `src/data/split.py` — the single authoritative splitter — using the method,
grouping column and seed in `configs/data.yaml`. Never call a splitter from
`sklearn` directly here: models validated on different folds cannot be compared.

In [ ]:
from src.data.split import train_test_split

train_df, test_df = train_test_split(df, cfg)
print(f"train: {len(train_df):,}   test: {len(test_df):,}"
      f"   ({len(test_df) / len(df):.1%} test)")

## 11. Leakage check

Cheap, and it catches the most expensive class of bug in the project: results
that look good and are wrong.

In [ ]:
from src.data.split import assert_no_group_leakage

assert_no_group_leakage(train_df, test_df, cfg)
print("no group appears in both splits")

# Sanity check the split is representative — a large distribution gap between
# train and test usually means the grouping column carries structure.
pd.DataFrame({
    "train": train_df[cfg.target].describe(),
    "test": test_df[cfg.target].describe(),
})

## 12. Persist and version

Write both splits, then track them with DVC so the exact contents are pinned to
this Git commit:

```bash
dvc add data/processed/train.parquet data/processed/test.parquet
git add data/processed/*.dvc configs/data.yaml
git commit -m "Clean and split dataset"
```

Or let the pipeline do it: `task dvc:repro` runs the same steps through
`src/data/clean.py`.

In [ ]:
from src.data.load import save_processed

save_processed(train_df, cfg.train_path)
save_processed(test_df, cfg.test_path)
print(f"wrote {cfg.train_path} and {cfg.test_path}")

## 13. Handoff checklist

Before moving to a `02x` notebook:

- [ ] Every filter applied here is deterministic and model-agnostic
- [ ] Every hard bound has a documented `source` in `configs/data.yaml`
- [ ] No imputation, scaling, encoding or statistical outlier removal happened
- [ ] The leakage assertion passed
- [ ] Both splits are written and `dvc add`-ed
- [ ] The cleaning audit is recorded

**The test split is now frozen. Nothing reads it again until notebook 03.**